# World Cup Upset Classifier
Trains an XGBoost model on 150 years of international football results to predict upset probability.

**Features**: all-time ELO diff, 3-year form ELO diff, confederation context, venue, match stakes  
**Targets**: binary upset flag + continuous upset score  
**Evaluation**: Brier score, ROC-AUC, calibration curve, SHAP explainability  

Prerequisite: run `elo_engine.py` first to generate `classifier_features.csv`

In [ ]:
# Install dependencies if needed
# !pip install xgboost shap scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    brier_score_loss, roc_auc_score, classification_report,
    RocCurveDisplay, CalibrationDisplay
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

plt.rcParams.update({'figure.dpi': 130, 'font.family': 'sans-serif'})
print('All imports OK')

## 1. Load & inspect the feature matrix

In [ ]:
df = pd.read_csv('classifier_features.csv', parse_dates=['date'])
print(f'Rows: {len(df):,}   Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# Sanity check: upsets should show negative elo_diff (underdog won)
print('=== Mean feature values by upset/non-upset ===')
check_cols = ['elo_diff', 'form_elo_diff', 'pre_prob_home', 'conf_k_mult', 'same_conf']
print(df.groupby('target')[check_cols].mean().round(3))

print(f"\nUpset rate: {df['target'].mean():.1%}")
print(f"Total upsets: {df['target'].sum():,}")

In [ ]:
# Feature correlation heatmap
fig, ax = plt.subplots(figsize=(9, 7))
feature_cols = ['elo_diff', 'form_elo_diff', 'conf_k_mult', 'same_conf',
                'is_neutral', 'tournament_k', 'pre_prob_home']
corr = df[feature_cols + ['target']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature correlations (target = upset)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

## 2. Train / test split
We use **temporal splitting** — not random. Train on matches before 2010, test on 2010–2022 World Cups. This mirrors real-world use: you train on history, predict the future.

In [ ]:
FEATURE_COLS = [
    'elo_diff',          # all-time quality gap
    'form_elo_diff',     # 3-year form gap (recency)
    'conf_k_mult',       # confederation strength of the match pool
    'same_conf',         # both teams from same (potentially weaker) pool
    'is_neutral',        # neutral venue removes home advantage
    'tournament_k',      # match stakes (WC=60, qualifier=30, friendly=5)
    'pre_prob_home',     # derived win probability from all-time ELO
]
TARGET_COL = 'target'

# Temporal split: train < 2010, test >= 2010
# Using 2010 gives ~40,000 training rows and ~9,000 test rows
SPLIT_DATE = '2010-01-01'

train = df[df['date'] < SPLIT_DATE].dropna(subset=FEATURE_COLS)
test  = df[df['date'] >= SPLIT_DATE].dropna(subset=FEATURE_COLS)

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]

print(f'Train: {len(X_train):,} rows  ({y_train.mean():.1%} upsets)')
print(f'Test:  {len(X_test):,}  rows  ({y_test.mean():.1%} upsets)')

## 3. Baseline: logistic regression
Always start simple. This gives us a performance floor and interpretable coefficients.

In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipe.fit(X_train, y_train)

lr_probs = lr_pipe.predict_proba(X_test)[:, 1]
lr_brier = brier_score_loss(y_test, lr_probs)
lr_auc   = roc_auc_score(y_test, lr_probs)

print(f'Logistic Regression  |  Brier: {lr_brier:.4f}  |  AUC: {lr_auc:.4f}')

# Coefficients — lower is more predictive of an upset
coef_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'coefficient': lr_pipe.named_steps['clf'].coef_[0]
}).sort_values('coefficient')
print('\nCoefficients (negative = pushes toward upset):')
print(coef_df.to_string(index=False))

## 4. XGBoost classifier
`scale_pos_weight` balances the class imbalance (~80% non-upset, ~20% upset).

In [ ]:
# Class weight: ratio of negatives to positives
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'scale_pos_weight = {scale_pos_weight:.2f}  (neg={neg:,}, pos={pos:,})')

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=4,           # shallow trees → less overfitting on noisy football data
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    verbosity=0,
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

xgb_probs = xgb.predict_proba(X_test)[:, 1]
xgb_brier = brier_score_loss(y_test, xgb_probs)
xgb_auc   = roc_auc_score(y_test, xgb_probs)

print(f'\nXGBoost              |  Brier: {xgb_brier:.4f}  |  AUC: {xgb_auc:.4f}')
print(f'Logistic Regression  |  Brier: {lr_brier:.4f}  |  AUC: {lr_auc:.4f}')
print(f'\nXGBoost Brier improvement: {(lr_brier - xgb_brier) / lr_brier:.1%}')

## 5. Calibration: does the model's 30% really mean 30%?
A well-calibrated model is one you can actually trust. If it says 25% chance of an upset, that team should win roughly 1 in 4 times historically.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Calibration curves
ax = axes[0]
CalibrationDisplay.from_predictions(y_test, lr_probs,  n_bins=10, ax=ax,
                                     name='Logistic Regression', color='steelblue')
CalibrationDisplay.from_predictions(y_test, xgb_probs, n_bins=10, ax=ax,
                                     name='XGBoost', color='tomato')
ax.set_title('Calibration curves\n(closer to diagonal = better)', fontsize=12)
ax.legend(fontsize=10)

# ROC curves
ax2 = axes[1]
RocCurveDisplay.from_predictions(y_test, lr_probs,  ax=ax2,
                                  name=f'Logistic Reg  (AUC={lr_auc:.3f})', color='steelblue')
RocCurveDisplay.from_predictions(y_test, xgb_probs, ax=ax2,
                                  name=f'XGBoost       (AUC={xgb_auc:.3f})', color='tomato')
ax2.set_title('ROC curves', fontsize=12)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('calibration_roc.png', bbox_inches='tight')
plt.show()

## 6. SHAP — why did the model predict an upset?
SHAP values explain each prediction. This is the resume-worthy part: you can point to any match and say *"the model flagged this as an upset risk because of X and Y."*

In [ ]:
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

# Global feature importance
plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, X_test, plot_type='bar',
                  feature_names=FEATURE_COLS, show=False)
plt.title('SHAP feature importance — global', fontsize=13)
plt.tight_layout()
plt.savefig('shap_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# Beeswarm: shows direction AND magnitude of each feature
plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, X_test,
                  feature_names=FEATURE_COLS, show=False)
plt.title('SHAP beeswarm — feature impact direction', fontsize=13)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', bbox_inches='tight')
plt.show()

## 7. Backtesting on World Cups only
The real test: how does the model perform on just the high-stakes World Cup matches it was never trained on?

In [ ]:
wc_test = test[test['tournament'].str.contains('FIFA World Cup', na=False)
               & ~test['tournament'].str.contains('qualif', case=False, na=False)]

X_wc = wc_test[FEATURE_COLS].dropna()
y_wc = wc_test.loc[X_wc.index, TARGET_COL]

wc_probs = xgb.predict_proba(X_wc)[:, 1]
wc_brier = brier_score_loss(y_wc, wc_probs)
wc_auc   = roc_auc_score(y_wc, wc_probs)

print(f'World Cup holdout   |  Brier: {wc_brier:.4f}  |  AUC: {wc_auc:.4f}')
print(f'All-test set        |  Brier: {xgb_brier:.4f}  |  AUC: {xgb_auc:.4f}')
print(f'\nWC matches in test set: {len(X_wc):,}  |  Upset rate: {y_wc.mean():.1%}')

In [ ]:
# Table: biggest predicted upsets that actually happened
wc_results = wc_test.loc[X_wc.index].copy()
wc_results['upset_prob'] = (wc_probs * 100).round(1)

# Matches the model thought were upsets AND that actually were
true_upsets = wc_results[wc_results['target'] == 1].sort_values('upset_prob', ascending=False)

display_cols = ['date', 'home_team', 'away_team', 'elo_diff', 'upset_prob']
print('=== Biggest predicted upsets (that actually happened) ===')
print(true_upsets[display_cols].head(15).to_string(index=False))

In [ ]:
# Misses: upsets the model didn't see coming (low upset_prob but target=1)
missed = wc_results[wc_results['target'] == 1].sort_values('upset_prob')
print('=== Upsets the model missed (low predicted probability) ===')
print(missed[display_cols].head(10).to_string(index=False))

## 8. Upset probability for any upcoming match
Plug in two teams and get a probability. This is the interactive piece for your portfolio.

In [ ]:
from elo_engine import EloEngine, get_k_factor, confederation_k_multiplier, get_confederation
import pandas as pd

# Load and fit the engine (needed for current ratings)
results_df = pd.read_csv('results.csv')
engine = EloEngine()
engine.fit(results_df)

def predict_upset_probability(
    home_team: str,
    away_team: str,
    neutral: bool = True,
    tournament: str = 'FIFA World Cup',
) -> dict:
    """
    Given two teams and match context, returns:
      - win probabilities for each team
      - upset probability (chance that the lower-rated team wins)
      - which team is the underdog
    """
    r_home = engine.rating(home_team)
    r_away = engine.rating(away_team)
    home_advantage = 0 if neutral else 100
    r_home_eff = r_home + home_advantage

    from elo_engine import expected_score
    p_home = expected_score(r_home_eff, r_away)
    p_away = 1 - p_home

    # Build feature vector matching training columns
    elo_diff       = r_home - r_away
    conf_mult      = confederation_k_multiplier(home_team, away_team)
    same_conf      = int(get_confederation(home_team) == get_confederation(away_team)
                         and get_confederation(home_team) is not None)
    tournament_k   = get_k_factor(tournament) * conf_mult

    # form_elo_diff not available in real-time without re-running the window;
    # use elo_diff as a proxy (or pre-compute from classifier_features.csv)
    form_elo_diff  = elo_diff

    features = pd.DataFrame([{
        'elo_diff':       elo_diff,
        'form_elo_diff':  form_elo_diff,
        'conf_k_mult':    conf_mult,
        'same_conf':      same_conf,
        'is_neutral':     int(neutral),
        'tournament_k':   tournament_k,
        'pre_prob_home':  p_home,
    }])

    upset_prob = xgb.predict_proba(features)[0][1]

    underdog   = away_team if p_home > 0.5 else home_team
    favorite   = home_team if p_home > 0.5 else away_team

    return {
        'favorite':          favorite,
        'underdog':          underdog,
        f'p_win_{home_team}': round(p_home * 100, 1),
        f'p_win_{away_team}': round(p_away * 100, 1),
        'upset_probability': f'{upset_prob * 100:.1f}%',
        'elo_home':          round(r_home, 1),
        'elo_away':          round(r_away, 1),
    }


# Try it out
for matchup in [
    ('Brazil',  'Germany',  True,  'FIFA World Cup'),
    ('Japan',   'Spain',    True,  'FIFA World Cup'),   # happened in 2022!
    ('England', 'France',   False, 'FIFA World Cup'),
    ('Saudi Arabia', 'Argentina', True, 'FIFA World Cup'),  # 2022 upset
]:
    result = predict_upset_probability(*matchup)
    print(f"\n{matchup[0]} vs {matchup[1]}")
    for k, v in result.items():
        print(f'  {k:<25} {v}')

## 9. All-time upset leaderboard
The headline visualization for your portfolio — the most shocking results in football history, ranked by the model.

In [ ]:
# Add model predictions to the full history
all_features = df[FEATURE_COLS].dropna()
df.loc[all_features.index, 'model_upset_prob'] = xgb.predict_proba(all_features)[:, 1]

# Filter: actual upsets only, World Cup matches only
wc_upsets = df[
    (df['target'] == 1) &
    df['tournament'].str.contains('FIFA World Cup', na=False) &
    ~df['tournament'].str.contains('qualif', case=False, na=False)
].copy()

wc_upsets['match'] = wc_upsets.apply(
    lambda r: f"{r['home_team']} vs {r['away_team']}", axis=1
)

top_upsets = wc_upsets.sort_values('model_upset_prob', ascending=False).head(20)

# Plot
fig, ax = plt.subplots(figsize=(11, 8))
bars = ax.barh(
    top_upsets['match'] + '  (' + top_upsets['date'].dt.year.astype(str) + ')',
    top_upsets['model_upset_prob'] * 100,
    color=plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(top_upsets)))
)

ax.set_xlabel('Model upset probability (%)', fontsize=11)
ax.set_title('Greatest World Cup upsets of all time\n(ranked by XGBoost upset probability)', fontsize=13)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('top_upsets.png', dpi=150, bbox_inches='tight')
plt.show()